# Full instrument: 30 analyser channels

Use the uninterrupted FANTASTIC elliptic guide and a 2–4 Å pulsed source.
Run from a fresh kernel, in cell order.


In [12]:
from pathlib import Path
import mcstasscript as ms
import os, sys, shutil, subprocess
from pathlib import Path
from mcstasscript.interface.functions import Configurator

prefix = Path(sys.prefix)
guides = list(prefix.rglob("Guide.comp"))
mcstas_path = guides[0].parent.parent
config = Configurator()
config.set_mcstas_path(str(mcstas_path))
config.set_mcrun_path(str(Path(shutil.which("mcrun")).parent))
os.environ["MCSTAS"] = str(mcstas_path)

print("Python:", sys.executable)
print("McStas library:", mcstas_path)

Python: /home/kyriakos2013/miniforge3/envs/mcstas/bin/python
McStas library: /home/kyriakos2013/miniforge3/envs/mcstas/share/mcstas/resources


In [13]:


instr = ms.McStas_instr(
    "full_instrument",
    author="McStasScript example",
    origin="Teaching example",
)
mpi=10

In [14]:
pulse_length = 400.0e-6
instr.add_parameter("pulse_length", value=pulse_length,
                    comment="Proton pulse duration for the moderator model [s]")
moderator_tau = 170.0e-6
instr.add_parameter("moderator_tau", value=moderator_tau,
                    comment="Cold moderator decay time used by Source_pulsed [s]")
moderator_n = 5.0
instr.add_parameter("moderator_n", value=moderator_n,
                    comment="Cold moderator pulse-shape parameter used by Source_pulsed")
moderator_time_center = 0.5*pulse_length + moderator_tau*(moderator_n+1.0)/moderator_n
source_frequency = 50.0
instr.add_parameter("source_frequency", value=source_frequency,
                    comment="Source repetition rate [Hz]")
lambda_min = 2.0
instr.add_parameter("lambda_min", value=lambda_min,
                    comment="Minimum source wavelength [Angstrom]")
lambda_max = 4.0
instr.add_parameter("lambda_max", value=lambda_max,
                    comment="Maximum source wavelength [Angstrom]")


source_t_max = 0.004
instr.add_parameter("double", "source_t_max", value=source_t_max,
                    comment="End of Source_pulsed emission-time sampling window [s]")

# The uninterrupted elliptic guide f
guide_parameters = {
    "linx": (0.611423, "Horizontal upstream focal distance [m]"),
    "liny": (1.04627, "Vertical upstream focal distance [m]"),
    "loutx": (1.68246, "Horizontal downstream focal distance [m]"),
    "louty": (1.94531, "Vertical downstream focal distance [m]"),
    "guide_length": (35.0, "Length of the uninterrupted elliptic guide [m]"),
    "guide_xwidth": (0.12, "Horizontal guide dimension at mid-length [m]"),
    "guide_yheight": (0.12, "Vertical guide dimension at mid-length [m]"),
    "d_source_guide": (0.5, "Source-to-guide entrance distance [m]"),
    "d_guide_sample": (1.5, "Guide exit-to-sample distance [m]"),
}
for name, (value, comment) in guide_parameters.items():
    instr.add_parameter("double", name, value=value, comment=comment)


In [15]:
source = instr.add_component("source", "Source_pulsed")
source.xwidth = 0.042
source.yheight = 0.03
source.Lmin = "lambda_min"
source.Lmax = "lambda_max"
source.t_min = 0.0
source.t_max = "source_t_max"
source.dist = "d_source_guide"
source.focus_xw = 0.04
source.focus_yh = 0.04
source.freq = "source_frequency"
source.t_pulse = "pulse_length"
source.T1 = 60.0
source.I1 = "1.75e12/source_frequency"
source.tau1 = "moderator_tau"
source.n_mod = "moderator_n"
source.I_um = "3.82e10/source_frequency"
source.tau_um = 12.0e-6
source.n_um = 5.0
source.chi_um = 0.9
source.kap_um = 2.2

guide = instr.add_component(
    "guide", "Elliptic_guide_gravity",
    AT=[0, 0, "d_source_guide"], RELATIVE="source"
)
guide.xwidth = "guide_xwidth"
guide.yheight = "guide_yheight"
guide.l = "guide_length"
guide.linxw = "linx"
guide.loutxw = "loutx"
guide.linyh = "liny"
guide.loutyh = "louty"
guide.dimensionsAt = '"mid"'
guide.option = '"ellipse"'
guide.R0 = 0.99
guide.Qc = 0.0218
guide.alpha = 6.07
guide.m = 4
guide.W = 0.003

In [16]:
guide_out = instr.add_component(
    "guide_out", "Arm", AT=[0, 0, "guide_length"], RELATIVE="guide"
)
sample_position = instr.add_component(
    "sample_position", "Arm",
    AT=[0, 0, "d_guide_sample"], RELATIVE="guide_out"
)


## Thirty analyser channels



In [17]:
channel_parameters = {
    "lambda_f": (4.0, "Analyser-selected wavelength [Angstrom]"),
    "channel_angle": (30.0, "Sample-to-analyser direction [degrees]"),
    "d_sample_analyser": (0.858, "Sample-to-analyser distance [m]"),
    "analyser_d": (3.357, "Analyser lattice spacing [Angstrom]"),
    "analyser_width": (0.07, "Total analyser width [m]"),
    "analyser_height": (0.03, "Total analyser height [m]"),
    "analyser_mosaic_h": (30.0, "Horizontal mosaic [arcmin]"),
    "analyser_mosaic_v": (30.0, "Vertical mosaic [arcmin]"),
    "analyser_NH": (11, "Horizontal blade count"),
    "analyser_NV": (11, "Vertical blade count"),
    "analyser_gap": (0.0005, "Blade gap [m]"),
    "analyser_detector_distance": (0.5, "FANTASTIC L2: separation along fixed detector arm [m]"),
    "detector_x": (2*0.75*0.0254, "Combined tube width [m]"),
    "detector_y": (22*0.0254, "Tube height [m]"),
    "sample_radius": (0.01, "Sample radius [m]"),
    "sample_height": (0.02, "Sample height [m]"),
    "sample_rotation_x": (0.0, "Sample x rotation [degrees]"),
    "sample_rotation_y": (0.0, "Sample y rotation [degrees]"),
    "sample_rotation_z": (0.0, "Sample z rotation [degrees]"),
}

channel_parameters["sample_E_tun"] = (2.5, "Tunneling energy transfer Ei - Ef [meV]")

channel_parameters["analyser_focusing"] = (1, "1: FANTASTIC focusing radii; 0: flat analyser")

for name, (value, description) in channel_parameters.items():
    parameter_type = "int" if name in ("analyser_NH", "analyser_NV") else "double"
    instr.add_parameter(parameter_type, name, value=value, comment=description)

In [18]:

for name in ("A5", "analyser_RH", "analyser_RV", "L_primary_channel", "tube_tmax",
             "detector_deflection", "detector_ray_path", "tube_center_y",
             "detector_hit_x", "detector_hit_y", "detector_hit_z"):
    instr.add_declare_var("double", name)

instr.add_declare_var("char", "tube_options", array=1024)

instr.append_initialize(r"""

A5 = RAD2DEG*asin(lambda_f/(2.0*analyser_d));
detector_deflection = 2.0*A5;


// Fixed -90 degree detector arm
detector_ray_path = analyser_detector_distance/sin(DEG2RAD*detector_deflection);
tube_center_y = -analyser_detector_distance/tan(DEG2RAD*detector_deflection);
detector_hit_x = (d_sample_analyser - tube_center_y)*sin(DEG2RAD*channel_angle);
detector_hit_y = -analyser_detector_distance;
detector_hit_z = (d_sample_analyser - tube_center_y)*cos(DEG2RAD*channel_angle);

// Flat analyser
analyser_RH = -1;
analyser_RV = -1;

// Detector time window
L_primary_channel = d_source_guide + guide_length + d_guide_sample;
tube_tmax = fmax(1.0/source_frequency,
    source_t_max + L_primary_channel*lambda_max/3956.0346 +
    (d_sample_analyser + detector_ray_path)*lambda_f/3956.0346);

snprintf(tube_options, sizeof(tube_options),
    "cylinder, y, bins=20, t, limits=[0.01 0.05], bins=1000", tube_tmax);


""")


## 1. Add SQW sample

### Choose one sample model



## 2. Build the 30 analyser channels

Create all positioning arms first, followed by a contiguous analysers group. Their directions are offset + 4k, matching FANTASTIC; offset = 0 keeps the requested arc. Each analyser uses the matched FANTASTIC blade counts, mosaic, dimensions and focusing radii. GROUP lets a ray try the bank’s alternative analysers.


## 3. Thirty slits and sixty tube monitors


In [19]:
# instr.show_diagram()
# instr.write_full_instrument()
# %matplotlib widget
# #replace "kirikarios" with your username!
# import sys; sys.path.insert(0, '/home/kirikarios/.conda/envs/mcstas/share/mcstas/tools/Python/mcdisplay/plopp')
# import mcdisplopp; mcdisplopp.ui('/home/kirikarios/Documents/full_instrument.instr', 100, '')

In [20]:


# instr.settings(force_compile=True, ncount=1e9, mpi=mpi)
# data=instr.backengine()
# ms.make_sub_plot(data,log=True)

In [21]:
# ms.make_sub_plot(data,log=False)